# Model 2 - Extended Dynamic

This notebook extends Model 1 by adding inflation, unemployment, and internet usage. It keeps lag GDP as the core dynamic term and tests whether the extra indicators improve predictive performance.

**Formula**

`target_log_gdp_next_year ~ log_gdp_per_capita + log_population_total + life_expectancy_years + inflation_pct_clean + unemployment_pct_clean + internet_users_pct_clean`


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

pd.options.display.float_format = "{:,.4f}".format
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


In [ ]:
if "__file__" in globals():
    BASE_DIR = Path(__file__).resolve().parents[1]
else:
    BASE_DIR = Path("/Users/tonytony/Final Project")

DATA_PATH = BASE_DIR / "Data" / "Cleaned" / "panel_with_event_dummies_and_extra_drivers.csv"
OUTPUT_DIR = BASE_DIR / "Data" / "Cleaned"

TRAIN_END_YEAR = 2017
TEST_START_YEAR = 2018
TEST_END_YEAR = 2022

MODEL_NAME = "Model 2 - Extended Dynamic"
FORMULA = "target_log_gdp_next_year ~ log_gdp_per_capita + log_population_total + life_expectancy_years + inflation_pct_clean + unemployment_pct_clean + internet_users_pct_clean"
OUTPUT_PREFIX = "gdp_model_2_extended_dynamic"

print("Base dir:", BASE_DIR)
print("Data path:", DATA_PATH)
print("Output dir:", OUTPUT_DIR)


In [ ]:
def regression_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))

    nonzero_mask = actual != 0
    if nonzero_mask.any():
        mape = np.mean(
            np.abs((actual[nonzero_mask] - predicted[nonzero_mask]) / actual[nonzero_mask])
        ) * 100
    else:
        mape = np.nan

    sst = np.sum((actual - actual.mean()) ** 2)
    sse = np.sum((actual - predicted) ** 2)
    r2 = 1 - (sse / sst) if sst > 0 else np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_pct": mape,
        "R_squared": r2,
    }


def build_prediction_frame(model, df):
    pred_df = df.copy()
    pred_df["predicted_log_gdp_next_year"] = model.predict(pred_df)
    pred_df["actual_log_gdp_next_year"] = pred_df["target_log_gdp_next_year"]

    pred_df["predicted_gdp_next_year"] = np.exp(pred_df["predicted_log_gdp_next_year"])
    pred_df["actual_gdp_next_year"] = np.exp(pred_df["actual_log_gdp_next_year"])

    pred_df["feature_year"] = pred_df["year"].astype(int)
    pred_df["target_year"] = pred_df["feature_year"] + 1
    pred_df["absolute_error_usd"] = np.abs(
        pred_df["actual_gdp_next_year"] - pred_df["predicted_gdp_next_year"]
    )
    pred_df["absolute_percentage_error_pct"] = (
        pred_df["absolute_error_usd"] / pred_df["actual_gdp_next_year"]
    ) * 100
    pred_df["signed_percentage_error_pct"] = (
        (pred_df["predicted_gdp_next_year"] - pred_df["actual_gdp_next_year"])
        / pred_df["actual_gdp_next_year"]
    ) * 100
    return pred_df


def build_region_metrics(df):
    rows = []
    for region_name, region_df in df.groupby("wb_region"):
        metrics = regression_metrics(
            actual=region_df["actual_gdp_next_year"],
            predicted=region_df["predicted_gdp_next_year"],
        )
        rows.append(
            {
                "wb_region": region_name,
                "n_obs": len(region_df),
                **metrics,
            }
        )
    return pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)


def build_yearly_summary(df):
    rows = []
    for target_year, year_df in df.groupby("target_year"):
        rows.append(
            {
                "target_year": int(target_year),
                "n_obs": len(year_df),
                "actual_mean_gdp": year_df["actual_gdp_next_year"].mean(),
                "predicted_mean_gdp": year_df["predicted_gdp_next_year"].mean(),
                "pred_to_actual_ratio": (
                    year_df["predicted_gdp_next_year"].mean()
                    / year_df["actual_gdp_next_year"].mean()
                ),
                "mean_absolute_percentage_error_pct": year_df[
                    "absolute_percentage_error_pct"
                ].mean(),
            }
        )
    return pd.DataFrame(rows).sort_values("target_year").reset_index(drop=True)


def show_or_close_plot():
    if "ipykernel" in sys.modules:
        plt.show()
    else:
        plt.close()


In [ ]:
model_cols = [
    "country_name",
    "country_code",
    "wb_region",
    "year",
    "gdp_per_capita_usd",
    "log_gdp_per_capita",
    "population_total",
    "log_population_total",
    "life_expectancy_years",
    "inflation_pct_clean",
    "unemployment_pct_clean",
    "internet_users_pct_clean",
    "target_log_gdp_next_year",
]

panel_df = pd.read_csv(DATA_PATH, usecols=model_cols)
model_df = panel_df.dropna().copy()

numeric_cols = [
    "year",
    "gdp_per_capita_usd",
    "log_gdp_per_capita",
    "population_total",
    "log_population_total",
    "life_expectancy_years",
    "inflation_pct_clean",
    "unemployment_pct_clean",
    "internet_users_pct_clean",
    "target_log_gdp_next_year",
]
for col in numeric_cols:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

model_df = model_df.dropna().copy()
model_df["year"] = model_df["year"].astype(int)
model_df["country_name"] = model_df["country_name"].astype(str)
model_df["country_code"] = model_df["country_code"].astype(str)
model_df["wb_region"] = model_df["wb_region"].astype(str)

print("Model sample shape:", model_df.shape)
print("Countries:", model_df["country_code"].nunique())
print("Regions:", model_df["wb_region"].nunique())
print("Feature-year range:", int(model_df["year"].min()), "-", int(model_df["year"].max()))
print(
    "Target-year range:",
    int((model_df["year"] + 1).min()),
    "-",
    int((model_df["year"] + 1).max()),
)

model_df.head()


In [ ]:
train_df = model_df[model_df["year"] <= TRAIN_END_YEAR].copy()
test_df = model_df[
    (model_df["year"] >= TEST_START_YEAR) & (model_df["year"] <= TEST_END_YEAR)
].copy()

print("Train rows:", train_df.shape[0])
print("Test rows:", test_df.shape[0])
print("Train countries:", train_df["country_code"].nunique())
print("Test countries:", test_df["country_code"].nunique())
print(
    "Train feature years:",
    int(train_df["year"].min()),
    "-",
    int(train_df["year"].max()),
)
print(
    "Test feature years:",
    int(test_df["year"].min()),
    "-",
    int(test_df["year"].max()),
)
print(
    "Test target years:",
    int((test_df["year"] + 1).min()),
    "-",
    int((test_df["year"] + 1).max()),
)


In [ ]:
fitted_model = smf.ols(formula=FORMULA, data=train_df).fit(cov_type="HC3")

print(MODEL_NAME)
print("Formula:", FORMULA)
print("Train R-squared:", round(fitted_model.rsquared, 4))
print(fitted_model.summary())


In [ ]:
coef_ci = fitted_model.conf_int()
coef_table = pd.DataFrame(
    {
        "term": fitted_model.params.index,
        "coefficient": fitted_model.params.values,
        "std_error": fitted_model.bse.values,
        "t_value": fitted_model.tvalues.values,
        "p_value": fitted_model.pvalues.values,
        "ci_lower": coef_ci[0].values,
        "ci_upper": coef_ci[1].values,
    }
).round(6)

coef_table


In [ ]:
train_predictions_df = build_prediction_frame(fitted_model, train_df)
test_predictions_df = build_prediction_frame(fitted_model, test_df)

metrics_df = pd.DataFrame(
    [
        {
            "split": "train",
            "scale": "level_gdp_usd",
            "model": MODEL_NAME,
            "n_obs": len(train_predictions_df),
            **regression_metrics(
                actual=train_predictions_df["actual_gdp_next_year"],
                predicted=train_predictions_df["predicted_gdp_next_year"],
            ),
        },
        {
            "split": "train",
            "scale": "log_gdp",
            "model": MODEL_NAME,
            "n_obs": len(train_predictions_df),
            **regression_metrics(
                actual=train_predictions_df["actual_log_gdp_next_year"],
                predicted=train_predictions_df["predicted_log_gdp_next_year"],
            ),
        },
        {
            "split": "test",
            "scale": "level_gdp_usd",
            "model": MODEL_NAME,
            "n_obs": len(test_predictions_df),
            **regression_metrics(
                actual=test_predictions_df["actual_gdp_next_year"],
                predicted=test_predictions_df["predicted_gdp_next_year"],
            ),
        },
        {
            "split": "test",
            "scale": "log_gdp",
            "model": MODEL_NAME,
            "n_obs": len(test_predictions_df),
            **regression_metrics(
                actual=test_predictions_df["actual_log_gdp_next_year"],
                predicted=test_predictions_df["predicted_log_gdp_next_year"],
            ),
        },
    ]
).round(4)

metrics_df


In [ ]:
region_metrics_df = build_region_metrics(test_predictions_df).round(4)
region_metrics_df


In [ ]:
bias_summary_df = pd.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "n_obs": len(test_predictions_df),
            "actual_mean_gdp": test_predictions_df["actual_gdp_next_year"].mean(),
            "predicted_mean_gdp": test_predictions_df["predicted_gdp_next_year"].mean(),
            "actual_median_gdp": test_predictions_df["actual_gdp_next_year"].median(),
            "predicted_median_gdp": test_predictions_df["predicted_gdp_next_year"].median(),
            "mean_absolute_percentage_error_pct": test_predictions_df[
                "absolute_percentage_error_pct"
            ].mean(),
            "mean_signed_percentage_error_pct": test_predictions_df[
                "signed_percentage_error_pct"
            ].mean(),
            "median_signed_percentage_error_pct": test_predictions_df[
                "signed_percentage_error_pct"
            ].median(),
            "pred_to_actual_mean_ratio": (
                test_predictions_df["predicted_gdp_next_year"].mean()
                / test_predictions_df["actual_gdp_next_year"].mean()
            ),
        }
    ]
).round(4)

bias_summary_df


In [ ]:
yearly_summary_df = build_yearly_summary(test_predictions_df).round(4)
yearly_summary_df


In [ ]:
year_plot_df = (
    test_predictions_df.groupby("target_year", as_index=False)[
        ["actual_gdp_next_year", "predicted_gdp_next_year"]
    ]
    .mean()
    .sort_values("target_year")
)

plt.figure(figsize=(12, 6))
plt.plot(
    year_plot_df["target_year"],
    year_plot_df["actual_gdp_next_year"],
    marker="o",
    linewidth=2.5,
    label="Actual GDP per Capita",
)
plt.plot(
    year_plot_df["target_year"],
    year_plot_df["predicted_gdp_next_year"],
    marker="o",
    linewidth=2.5,
    linestyle="--",
    label="Predicted GDP per Capita",
)
plt.title(f"{MODEL_NAME}: Mean Actual vs Predicted GDP per Capita")
plt.xlabel("Target Year")
plt.ylabel("GDP per Capita (US$)")
plt.legend()
plt.tight_layout()
show_or_close_plot()


In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(
    test_predictions_df["actual_gdp_next_year"],
    test_predictions_df["predicted_gdp_next_year"],
    alpha=0.6,
)

diag_min = min(
    test_predictions_df["actual_gdp_next_year"].min(),
    test_predictions_df["predicted_gdp_next_year"].min(),
)
diag_max = max(
    test_predictions_df["actual_gdp_next_year"].max(),
    test_predictions_df["predicted_gdp_next_year"].max(),
)
plt.plot([diag_min, diag_max], [diag_min, diag_max], color="red", linestyle="--")
plt.title(f"{MODEL_NAME}: Actual vs Predicted GDP per Capita")
plt.xlabel("Actual GDP per Capita (US$)")
plt.ylabel("Predicted GDP per Capita (US$)")
plt.tight_layout()
show_or_close_plot()


In [ ]:
output_files = {
    "metrics": OUTPUT_DIR / f"{OUTPUT_PREFIX}_metrics.csv",
    "coefficients": OUTPUT_DIR / f"{OUTPUT_PREFIX}_coefficients.csv",
    "test_predictions": OUTPUT_DIR / f"{OUTPUT_PREFIX}_test_predictions.csv",
    "region_metrics": OUTPUT_DIR / f"{OUTPUT_PREFIX}_region_metrics.csv",
    "yearly_summary": OUTPUT_DIR / f"{OUTPUT_PREFIX}_yearly_summary.csv",
    "bias_summary": OUTPUT_DIR / f"{OUTPUT_PREFIX}_bias_summary.csv",
}

metrics_df.to_csv(output_files["metrics"], index=False)
coef_table.to_csv(output_files["coefficients"], index=False)
test_predictions_df.to_csv(output_files["test_predictions"], index=False)
region_metrics_df.to_csv(output_files["region_metrics"], index=False)
yearly_summary_df.to_csv(output_files["yearly_summary"], index=False)
bias_summary_df.to_csv(output_files["bias_summary"], index=False)

for label, path in output_files.items():
    print(f"{label}: {path}")
